# ETL Data Architecture in Spark Streaming
### Báo cáo Lý thuyết & Thực nghiệm Kiến trúc — Part A.1.2

> **Học phần:** Big Data / Xử lý Dữ liệu lớn  
> **Đề tài:** Real-Time Streaming with Kafka & PySpark Structured Streaming  
> **Nhóm thực hiện:** Nhóm 5  
> **Chủ đề nghiên cứu:** Kiến trúc dữ liệu ETL trong Spark Streaming (ETL Data Architecture in Spark Streaming)  

---

### Yêu cầu trọng tâm của đề tài (Part A – 30 marks):

1. **Giải thích vai trò và tầm quan trọng của quy trình ETL** (Extract – Transform – Load) trong các hệ thống xử lý dữ liệu liên tục thời gian thực (Continuous / Real-time Processing).
2. **Cơ chế hỗ trợ toàn diện của Spark Streaming đối với quy trình ETL**, thể hiện qua bốn trụ cột kiến trúc:
   - **Schema enforcement** (Ràng buộc lược đồ dữ liệu)
   - **Stateful transformations** (Các phép biến đổi duy trì trạng thái theo cửa sổ thời gian và watermark)
   - **Fault tolerance** (Khả năng chịu lỗi và tự phục hồi qua Write-Ahead Log + Checkpointing)
   - **End-to-end exactly-once semantics** (Ngữ nghĩa đảm bảo xử lý chính xác đúng một lần trên toàn luồng)
3. **Tổng quan tài liệu nghiên cứu (Literature Review):** Tóm tắt, phân tích và trích dẫn chuẩn mực một công trình khoa học tiêu biểu về tối ưu hóa kiến trúc Streaming ETL.


## 1. ETL là gì &amp; tại sao quan trọng trong xử lý liên tục

**ETL** là quy trình chuẩn hóa dữ liệu gồm 3 giai đoạn:

- **Extract (Trích xuất):** lấy dữ liệu thô từ nguồn &#8212; trong hệ streaming là một *dòng sự kiện vô hạn* (Kafka topic, log, clickstream).
- **Transform (Biến đổi):** parse, làm sạch, ép kiểu, lọc, làm giàu (enrich), tính toán tổng hợp (aggregate) theo thời gian.
- **Load (Nạp):** ghi kết quả đã chuẩn hóa ra nơi lưu trữ / phục vụ (data lake, warehouse, dashboard, API).



<div style="overflow-x:auto; background:#ffffff; border:1px solid #e2e8f0; border-radius:12px; padding:18px 10px;">
<svg viewBox="0 0 860 260" width="100%" style="max-width:860px; font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;">
  <rect x="0" y="0" width="860" height="260" fill="#ffffff"/>
  <text x="430" y="28" text-anchor="middle" font-size="16" font-weight="700" fill="#0f172a">ETL trong xử lý dữ liệu liên tục (continuous / streaming)</text>

  <!-- EXTRACT -->
  <rect x="30" y="70" width="210" height="120" rx="10" fill="#dbeafe" stroke="#1e40af" stroke-width="1.5"/>
  <text x="135" y="98" text-anchor="middle" font-size="15" font-weight="700" fill="#1e3a8a">EXTRACT</text>
  <text x="135" y="122" text-anchor="middle" font-size="12" fill="#1e3a8a">Đọc dòng sự kiện vô hạn</text>
  <text x="135" y="140" text-anchor="middle" font-size="12" fill="#1e3a8a">Kafka / File / Socket</text>
  <text x="135" y="162" text-anchor="middle" font-size="11" fill="#334155">spark.readStream</text>

  <!-- TRANSFORM -->
  <rect x="325" y="70" width="210" height="120" rx="10" fill="#dcfce7" stroke="#166534" stroke-width="1.5"/>
  <text x="430" y="98" text-anchor="middle" font-size="15" font-weight="700" fill="#166534">TRANSFORM</text>
  <text x="430" y="122" text-anchor="middle" font-size="12" fill="#166534">Parse JSON, ép kiểu, làm sạch</text>
  <text x="430" y="140" text-anchor="middle" font-size="12" fill="#166534">Window + Watermark + Agg</text>
  <text x="430" y="162" text-anchor="middle" font-size="11" fill="#334155">select / withColumn / groupBy</text>

  <!-- LOAD -->
  <rect x="620" y="70" width="210" height="120" rx="10" fill="#fef3c7" stroke="#92400e" stroke-width="1.5"/>
  <text x="725" y="98" text-anchor="middle" font-size="15" font-weight="700" fill="#92400e">LOAD</text>
  <text x="725" y="122" text-anchor="middle" font-size="12" fill="#92400e">Ghi ra sink (bền vững)</text>
  <text x="725" y="140" text-anchor="middle" font-size="12" fill="#92400e">Parquet / Delta / Console</text>
  <text x="725" y="162" text-anchor="middle" font-size="11" fill="#334155">writeStream + checkpoint</text>

  <!-- arrows -->
  <defs>
    <marker id="ah" markerWidth="10" markerHeight="10" refX="7" refY="3" orient="auto" markerUnits="strokeWidth">
      <path d="M0,0 L7,3 L0,6 Z" fill="#475569"/>
    </marker>
  </defs>
  <line x1="245" y1="130" x2="320" y2="130" stroke="#475569" stroke-width="2.5" marker-end="url(#ah)"/>
  <line x1="540" y1="130" x2="615" y2="130" stroke="#475569" stroke-width="2.5" marker-end="url(#ah)"/>

  <text x="430" y="228" text-anchor="middle" font-size="12" fill="#64748b">Luồng chạy liên tục, không ngừng &#8212; mỗi micro-batch lặp lại E → T → L trên dữ liệu mới</text>
  <text x="430" y="247" text-anchor="middle" font-size="11" fill="#94a3b8">Kiểm soát đúng đắn bằng checkpoint (WAL + state) → exactly-once</text>
</svg>
</div>


### 1.1. So sánh kiến trúc: Xử lý theo lô (Batch ETL) và Xử lý liên tục (Streaming ETL)

Trong khi các hệ thống xử lý theo lô (chẳng hạn như Apache Airflow kích hoạt định kỳ) phù hợp cho việc tổng hợp dữ liệu tĩnh theo khung thời gian dài, thì các bài toán nghiệp vụ hiện đại (**phát hiện gian lận - fraud detection, cảnh báo hệ thống, cá nhân hóa trải nghiệm theo thời gian thực**) đòi hỏi thời gian phản hồi ở mức dưới giây (sub-second). Nếu phải chờ đợi tích lũy theo lô, giá trị tức thời của thông tin (ví dụ: tín hiệu người dùng chuẩn bị rời bỏ giỏ hàng) sẽ hoàn toàn mất đi trước khi hệ thống kịp kích hoạt hành động.

<table>
<thead><tr><th>Tiêu chí so sánh</th><th>Batch ETL (Xử lý theo lô)</th><th>Streaming ETL (Xử lý liên tục)</th></tr></thead>
<tbody>
<tr><td><b>Đặc tính dữ liệu đầu vào</b></td><td>Tập dữ liệu hữu hạn (Bounded Dataset) — tệp hoặc bảng cố định</td><td>Dòng sự kiện vô hạn (Unbounded Stream) — xuất hiện liên tục</td></tr>
<tr><td><b>Độ trễ xử lý (Latency)</b></td><td>Từ hàng phút đến hàng giờ hoặc hàng ngày</td><td>Từ mili-giây đến vài giây</td></tr>
<tr><td><b>Cơ chế kích hoạt</b></td><td>Theo lịch biểu định kỳ (Cron Schedule / Workflow DAG)</td><td>Liên tục theo micro-batch hoặc dòng sự kiện (Event-driven)</td></tr>
<tr><td><b>Quản lý trạng thái (State)</b></td><td>Không lưu trạng thái liên tục giữa các lần chạy (Stateless)</td><td>Trạng thái được duy trì xuyên suốt các batch (Windowing, Session state)</td></tr>
<tr><td><b>Xử lý dữ liệu đến muộn</b></td><td>Phải tái tính toán và nạp lại toàn bộ lô dữ liệu</td><td>Tự động tiếp nhận và cập nhật qua cơ chế Watermark</td></tr>
<tr><td><b>Ứng dụng tiêu biểu</b></td><td>Báo cáo tài chính cuối ngày, Data Warehouse định kỳ</td><td>Phát hiện gian lận thẻ, gợi ý sản phẩm thời gian thực</td></tr>
</tbody></table>

### 1.2. So sánh kiến trúc ETL và ELT trong kỷ nguyên dữ liệu lớn

<table>
<thead><tr><th>Tiêu chí kiến trúc</th><th>ETL (Extract → Transform → Load)</th><th>ELT (Extract → Load → Transform)</th></tr></thead>
<tbody>
<tr><td><b>Vị trí thực thi Transform</b></td><td>Thực hiện trực tiếp trong Engine tính toán phân tán (Spark) <i>trước khi</i> nạp vào nơi lưu trữ</td><td>Nạp dữ liệu thô vào Data Lake/Lakehouse trước, sau đó mới dùng sức mạnh của kho để biến đổi</td></tr>
<tr><td><b>Thời điểm làm sạch & chuẩn hóa</b></td><td>Thực hiện ngay trên đường truyền dữ liệu (On-the-fly)</td><td>Thực hiện sau đó thông qua các câu truy vấn SQL/dbt theo lô</td></tr>
<tr><td><b>Phạm vi áp dụng tối ưu</b></td><td>Đòi hỏi chất lượng dữ liệu nghiêm ngặt, chuẩn hóa schema sớm, độ trễ phản hồi thấp</td><td>Lưu trữ dữ liệu thô quy mô cực lớn, tài nguyên tính toán đàn hồi trong Cloud Warehouse</td></tr>
<tr><td><b>Lựa chọn trong đề tài của nhóm</b></td><td><b>✓ Lựa chọn cốt lõi:</b> Spark Structured Streaming thực hiện trích xuất, làm sạch và gán trọng số ngay khi đọc từ Kafka</td><td>Không tối ưu cho nhu cầu phản hồi tức thời của hệ thống gợi ý</td></tr>
</tbody></table>

> **Kết luận chuyên môn của nhóm:** Đối với bài toán *Phân tích hành vi phiên truy cập và gợi ý sản phẩm thời gian thực (Real-Time E-Commerce Session Tracker)*, **Streaming ETL** là sự lựa chọn duy nhất đáp ứng yêu cầu nghiệp vụ: hệ thống phải phát hiện kịp thời các khách hàng có ý định mua sắm cao (High-Intent) **ngay trong lúc phiên truy cập còn đang diễn ra**, điều mà các mô hình Batch hoặc ELT trễ hạn không thể đáp ứng được.

## 2. Spark Structured Streaming dưới vai trò một Streaming ETL Engine

Spark Structured Streaming xây dựng trên mô hình trừu tượng hóa dòng dữ liệu thành một **Bảng dữ liệu vô hạn (Unbounded Table)** liên tục được chèn thêm các bản ghi mới. Điểm vượt trội về mặt kiến trúc đối với quy trình ETL là: **đồng nhất hoàn toàn mã nguồn xử lý DataFrame cho cả hai hình thức Batch và Streaming** — chỉ khác biệt ở phương thức đọc (`read` ↔ `readStream`) và phương thức ghi (`write` ↔ `writeStream`). Bộ tối ưu hóa Catalyst Engine sẽ tự động chuyển đổi các phép toán khai báo thành kế hoạch thực thi tăng dần (Incremental Execution Plan).

**Ánh xạ các pha của chu trình ETL vào hệ thống Spark API:**

<table>
<thead><tr><th>Giai đoạn ETL</th><th>Nhiệm vụ kiến trúc</th><th>Hiện thực bằng Spark Structured Streaming API</th></tr></thead>
<tbody>
<tr><td><b>Extract</b></td><td>Trích xuất luồng sự kiện từ nguồn vô hạn</td><td><code>spark.readStream.format("kafka"/"json"/"rate").schema(...)</code></td></tr>
<tr><td><b>Transform (Stateless)</b></td><td>Phân giải, lọc dữ liệu lỗi, chuyển đổi kiểu và gán nhãn thuộc tính</td><td><code>select, filter, withColumn, from_json, cast</code></td></tr>
<tr><td><b>Transform (Stateful)</b></td><td>Tổng hợp dữ liệu theo cửa sổ thời gian, khử trùng lặp và duy trì trạng thái</td><td><code>withWatermark, groupBy(window(...)), join, dropDuplicates</code></td></tr>
<tr><td><b>Load</b></td><td>Nạp dữ liệu vào đích lưu trữ bền vững kèm đảm bảo ngữ nghĩa</td><td><code>writeStream.outputMode(...).option("checkpointLocation",...).start()</code></td></tr>
</tbody></table>

**Ba chế độ xuất dữ liệu (Output Mode) — Xác định nội dung ghi ra Sink qua mỗi chu kỳ xử lý:**

| Output Mode | Nội dung được đẩy ra hệ thống lưu trữ đích (Sink) | Trường hợp áp dụng tiêu biểu |
|---|---|---|
| **Append Mode** | Chỉ những dòng dữ liệu mới phát sinh (bất biến) của micro-batch hiện tại | Các truy vấn ETL không tổng hợp, hoặc truy vấn tổng hợp đã thỏa mãn điều kiện vượt ngưỡng Watermark |
| **Update Mode** | Chỉ những dòng kết quả tổng hợp có sự thay đổi giá trị trong micro-batch | Các bảng tổng hợp trạng thái cần cập nhật liên tục cho Dashboard hoặc hệ thống trực tuyến |
| **Complete Mode** | Toàn bộ bảng kết quả tổng hợp được ghi đè từ đầu qua mỗi micro-batch | Các bài toán thống kê tổng quát quy mô nhỏ, số lượng nhóm hữu hạn |

Đoạn mã sau đây khởi tạo SparkSession phục vụ cho toàn bộ các thực nghiệm kiến trúc trong notebook.

In [1]:
# ===== KHỞI TẠO SPARK SESSION CHO THỰC NGHIỆM KIẾN TRÚC ETL =====
import os, sys, json, time, random, shutil, glob

# Đồng bộ hóa môi trường Python giữa Driver và Worker để tránh xung đột phiên bản
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F          # Thư viện hàm xử lý biểu thức cột
from pyspark.sql.types import (                 # Thư viện định nghĩa kiểu dữ liệu cho Schema
    StructType, StructField, LongType, IntegerType, StringType, TimestampType)

spark = (
    SparkSession.builder
    .appName("PartA-ETL-Architecture")
    .master("local[2]")                               # Cấu hình thực thi cục bộ với 2 luồng tính toán
    .config("spark.sql.shuffle.partitions", "4")      # Tối ưu số lượng phân vùng shuffle cho môi trường thực nghiệm
    .config("spark.ui.enabled", "false")              # Vô hiệu hóa giao diện web UI để tập trung vào luồng xử lý
    .config("spark.ui.showConsoleProgress", "false")  # Tắt thanh tiến trình để giữ thông tin xuất sạch sẽ
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")               # Chỉ hiển thị thông báo lỗi cấp cao, ẩn các log cảnh báo không cần thiết

# Xuất thông tin định danh của môi trường tính toán
print("[Hệ thống] Spark version :", spark.version)
print("[Hệ thống] Master        :", spark.sparkContext.master)
print("[Hệ thống] Python driver :", sys.version.split()[0])

26/10/26 23:08:56 WARN Utils: Your hostname, MacBook-Pro-ca-Tun-Phong.local resolves to a loopback address: 127.0.0.1; using 192.168.1.140 instead (on interface en0)
26/10/26 23:08:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/26 23:08:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version : 3.5.9
Master        : local[2]
Python driver : 3.10.20


### 2.1. Phân tích sự khác biệt giữa Kafka Source và File Stream Source trong kiến trúc Extract

Trong khuôn khổ thực nghiệm độc lập của notebook Part A, nhóm áp dụng cơ chế **File Streaming Source** (tự động phát sinh các tệp JSON tuân thủ chính xác lược đồ của bộ dữ liệu Retail Rocket vào một thư mục đệm; mỗi tệp mới xuất hiện được xem như một micro-batch độc lập). Phương pháp này cho phép nhóm kiểm thử toàn diện logic xử lý ETL và cơ chế quản lý trạng thái trong Spark trước khi kết nối trực tiếp với cụm phân tán Apache Kafka ở Part B.

**So sánh cấu trúc dữ liệu trích xuất theo từng loại nguồn:**

| Nguồn dữ liệu (Source) | Dữ liệu trả về cho Spark | Các bước xử lý bắt buộc tiếp theo |
|---|---|---|
| **File Source (JSON/CSV)** | Các cột dữ liệu đã được định kiểu theo cấu trúc schema khai báo trước | Dữ liệu sẵn sàng đưa vào các phép biến đổi trực tiếp |
| **Kafka Source** | Khung bản ghi chuẩn chứa các trường metadata: `key`, `value`, `topic`, `partition`, `offset`, `timestamp` — trong đó trường nội dung chính `value` ở định dạng nhị phân (`BinaryType`) | Bắt buộc thực hiện chuyển đổi kiểu `CAST(value AS STRING)` kết hợp hàm `from_json(..., schema)` để phân giải thành các trường dữ liệu có kiểu |

> **Nhận xét kiến trúc của nhóm:** File Stream Source cung cấp dữ liệu cấu trúc trực tiếp, thuận tiện cho việc kiểm chứng thuật toán. Trong khi đó, Kafka Source cung cấp payload thô kèm theo thông tin vị trí `partition`/`offset` tin cậy — đây chính là yếu tố kỹ thuật then chốt giúp Spark thực hiện cơ chế đọc lại (Replayable) khi gặp sự cố, tạo tiền đề hiện thực hóa ngữ nghĩa *Exactly-Once* toàn diện (được nhóm chứng minh tại Mục 7).

In [2]:
# ===== SINH DỮ LIỆU THỰC NGHIỆM THEO CHUẨN LƯỢC ĐỒ RETAIL ROCKET =====
# Nhóm xây dựng hàm tạo dữ liệu mô phỏng chính xác cấu trúc dữ liệu thực tế
random.seed(42)                      # Cố định giá trị ngẫu nhiên để đảm bảo tính tái lập của kết quả thực nghiệm

BASE = "/tmp/retailrocket_demo"
LANDING = f"{BASE}/landing"          # Thư mục đệm đóng vai trò nguồn File Streaming
shutil.rmtree(BASE, ignore_errors=True)
os.makedirs(LANDING, exist_ok=True)

EPOCH0 = 1433221332117               # Mốc thời gian gốc (tương ứng thời điểm thực của tập dữ liệu Retail Rocket)
EVENTS = [("view", 80), ("addtocart", 15), ("transaction", 5)]   # Tỷ lệ phân bố các hành vi tương tác

def make_event(i):                   # Tạo một bản ghi sự kiện tuân thủ đặc tả nghiệp vụ
    ev = random.choices([e for e, _ in EVENTS], weights=[w for _, w in EVENTS])[0]
    return {
        "timestamp": EPOCH0 + i * 15000,                  # Khoảng cách thời gian 15 giây giữa các sự kiện
        "visitorid": random.randint(1, 12),               # Định danh người dùng (visitor ID)
        "event": ev,
        "itemid": random.randint(1000, 1008),             # Định danh sản phẩm
        "transactionid": (random.randint(1, 9999) if ev == "transaction" else None),
    }

# Tạo 3 tệp dữ liệu (mỗi tệp chứa 60 bản ghi JSON) nhằm mô phỏng dòng dữ liệu đến dần theo thời gian
idx = 0
for b in range(3):
    rows = [json.dumps(make_event(idx + k)) for k in range(60)]
    with open(f"{LANDING}/events_{b:03d}.json", "w") as f:
        f.write("\n".join(rows))
    idx += 60

print(f"[Thực nghiệm] Đã tạo thành công {len(os.listdir(LANDING))} tệp dữ liệu tại {LANDING}")
print("\n[Mẫu dữ liệu thô trích xuất từ tệp đầu tiên]:")
for line in open(sorted(glob.glob(f"{LANDING}/*.json"))[0]).read().splitlines()[:2]:
    print(" ", line)

Đã tạo 3 file tại /tmp/retailrocket_demo/landing

Ví dụ 2 bản ghi thô:
  {"timestamp": 1433221332117, "visitorid": 1, "event": "view", "itemid": 1004, "transactionid": null}
  {"timestamp": 1433221347117, "visitorid": 3, "event": "view", "itemid": 1001, "transactionid": null}


## 3. Extract — Cơ chế trích xuất dòng dữ liệu Streaming

Quy trình trích xuất (Extract) trong Spark Structured Streaming được nhóm tiến hành theo các bước chuẩn mực:

1. **Lựa chọn định dạng và nguồn (Source Format):** `spark.readStream.format("kafka" / "json" / ...)`.  
2. **Áp đặt lược đồ cấu trúc (Schema Enforcement):** Bước bắt buộc đối với luồng streaming nhằm ngăn chặn việc suy diễn schema tốn kém trên dòng dữ liệu vô hạn (chi tiết tại Mục 4).  
3. **Thiết lập tham số streaming (Stream Options):** Cấu hình `maxFilesPerTrigger` đối với File source (hoặc `subscribe`, `startingOffsets` đối với Kafka source).  
4. **Khởi tạo đối tượng Streaming DataFrame:** Thực hiện `.load()` hoặc `.json(path)`.  
5. **Phân giải payload nhị phân (áp dụng riêng cho Kafka):** Sử dụng `CAST(value AS STRING)` kết hợp hàm `from_json` theo schema đã định nghĩa.  

*Đặc tính kiến trúc:* Streaming DataFrame hoạt động theo cơ chế **Lazy Evaluation** — đối tượng này chỉ lưu trữ đồ thị tính toán logic (Logical Plan) và chỉ thực sự kích hoạt luồng đọc/xử lý khi phương thức `writeStream.start()` được gọi ở giai đoạn Load. Đoạn mã bên dưới hiện thực các bước trích xuất nêu trên:

In [3]:
# ===== EXTRACT: Trích xuất dòng dữ liệu từ thư mục đệm (File Stream Source) =====
# Định nghĩa lược đồ dữ liệu (Schema Contract) cho 5 trường thuộc tính cốt lõi
schema = StructType([
    StructField("timestamp",     LongType(),    True),   # Mốc thời gian phát sinh sự kiện (Epoch milliseconds)
    StructField("visitorid",     IntegerType(), True),   # Mã định danh khách truy cập
    StructField("event",         StringType(),  True),   # Loại sự kiện (view / addtocart / transaction)
    StructField("itemid",        IntegerType(), True),   # Mã định danh mặt hàng
    StructField("transactionid", IntegerType(), True),   # Mã giao dịch thanh toán
])

raw_stream = (
    spark.readStream
    .schema(schema)                      # Áp đặt schema bắt buộc cho luồng dữ liệu vô hạn
    .option("maxFilesPerTrigger", 1)     # Giới hạn xử lý 1 tệp trên mỗi micro-batch để quan sát tiến trình
    .json(LANDING)                       # Đọc các tệp JSON trong thư mục đệm
)

print("[Kiểm tra] Trạng thái Streaming DataFrame (isStreaming):", raw_stream.isStreaming)
print("[Lược đồ dữ liệu đã áp đặt]:")
raw_stream.printSchema()

Đây có phải streaming DataFrame? -> True
root
 |-- timestamp: long (nullable = true)
 |-- visitorid: integer (nullable = true)
 |-- event: string (nullable = true)
 |-- itemid: integer (nullable = true)
 |-- transactionid: integer (nullable = true)



## 4. Schema Enforcement — "Hợp đồng dữ liệu" (Data Contract) trong Streaming

Trong môi trường xử lý liên tục, dòng sự kiện phát sinh không ngừng và **hệ thống không thể tạm dừng để xử lý các xung đột cấu trúc một cách thủ công**. Vì vậy, việc áp đặt schema tường minh là nguyên tắc kiến trúc bắt buộc:

- **Spark không hỗ trợ suy diễn lược đồ (Schema Inference) cho luồng streaming:** Lý do là engine không thể quét trước toàn bộ dữ liệu vô hạn để phỏng đoán kiểu dữ liệu → nhóm bắt buộc phải khai báo cấu trúc schema từ đầu.
- **Đóng vai trò "Hợp đồng dữ liệu" (Data Contract):** Giúp kiểm soát chặt chẽ giao thức truyền thông giữa hệ thống sinh dữ liệu (Producer) và hệ thống tiếp nhận (Consumer). Mọi trường hợp sai lệch định dạng hoặc thiếu sót trường thông tin đều được phát hiện và xử lý ngay tại tầng nhập khẩu.
- **Xử lý bản ghi từ Kafka:** Dữ liệu payload từ Kafka là chuỗi byte/JSON string → sử dụng hàm `from_json(value, schema)` để phân rã chính xác thành các cột có kiểu dữ liệu tường minh.
- **Khả năng chịu lỗi với dữ liệu tha hóa:** Các bản ghi bị hỏng hoặc sai cú pháp JSON được Spark gán giá trị `null` thay vì gây dừng đột ngột (crash) toàn bộ công việc xử lý.

<table>
<thead><tr><th>Tiêu chí so sánh</th><th>Suy diễn lược đồ (Schema Inference)</th><th>Ràng buộc lược đồ (Schema Enforcement)</th></tr></thead>
<tbody>
<tr><td><b>Khả năng áp dụng trong Streaming</b></td><td>Không khả thi trên dòng dữ liệu vô hạn</td><td><b>Bắt buộc áp dụng ✓</b></td></tr>
<tr><td><b>Tính an toàn kiểu (Type Safety)</b></td><td>Dễ xảy ra hiện tượng trôi dạt lược đồ (Schema Drift) giữa các batch</td><td>Bảo đảm tính ổn định và nhất quán tuyệt đối</td></tr>
<tr><td><b>Hiệu năng tính toán</b></td><td>Tốn chi phí quét toàn bộ dữ liệu ban đầu</td><td>Tối ưu hiệu năng, không tiêu tốn tài nguyên quét trước</td></tr>
</tbody></table>

Đoạn mã dưới đây minh chứng cơ chế hoạt động của `from_json` trên chuỗi payload JSON (tương tự trường `value` nhận từ Kafka), đồng thời kiểm chứng khả năng tự cô lập lỗi khi gặp bản ghi sai định dạng:

In [4]:
# ===== MINH CHỨNG SCHEMA ENFORCEMENT & XỬ LÝ LỖI ĐỊNH DẠNG PAYLOAD =====
# Nhóm mô phỏng 3 bản ghi chuỗi payload (tương tự trường value từ Kafka); bản ghi thứ 3 cố tình tạo lỗi cú pháp JSON
sample = spark.createDataFrame([
    ('{"timestamp":1433221332117,"visitorid":257597,"event":"addtocart","itemid":355908,"transactionid":null}',),
    ('{"timestamp":1433221347117,"visitorid":99,"event":"view","itemid":1001,"transactionid":null}',),
    ('{ chuỗi JSON HỎNG - thiếu dấu ngoặc',),     # Bản ghi lỗi cấu trúc để kiểm tra độ tin cậy của engine
], ["value"])

# Áp dụng from_json để phân rã chuỗi JSON theo cấu trúc schema
parsed = sample.select(F.from_json("value", schema).alias("d")).select("d.*")
parsed.show(truncate=False)
print("[Kết quả thực nghiệm] Bản ghi sai cấu trúc được chuyển đổi thành các trường null an toàn, pipeline duy trì hoạt động bình thường.")

+-------------+---------+---------+------+-------------+
|timestamp    |visitorid|event    |itemid|transactionid|
+-------------+---------+---------+------+-------------+
|1433221332117|257597   |addtocart|355908|NULL         |
|1433221347117|99       |view     |1001  |NULL         |
|NULL         |NULL     |NULL     |NULL  |NULL         |
+-------------+---------+---------+------+-------------+

Bản ghi hỏng -> toàn bộ cột = null (pipeline KHÔNG sập).


## 5. Transform — Phân định phép biến đổi Không trạng thái (Stateless) và Có trạng thái (Stateful)

Giai đoạn Transform đóng vai trò trung tâm trong pipeline ETL. Trong Spark Structured Streaming, các phép biến đổi được phân chia thành hai nhóm chính với các đặc trưng kỹ thuật riêng biệt:

<table>
<thead><tr><th>Tiêu chí phân định</th><th>Biến đổi Không trạng thái (Stateless Transformation)</th><th>Biến đổi Có trạng thái (Stateful Transformation)</th></tr></thead>
<tbody>
<tr><td><b>Nguyên lý hoạt động</b></td><td>Mỗi bản ghi được xử lý hoàn toàn độc lập, không phụ thuộc vào các sự kiện quá khứ</td><td>Hệ thống phải duy trì và cập nhật trạng thái tích lũy qua nhiều micro-batch liên tiếp</td></tr>
<tr><td><b>Các thao tác tiêu biểu</b></td><td><code>select, filter, withColumn, from_json, cast</code></td><td><code>groupBy(window).agg</code>, <code>join</code> giữa các dòng, <code>dropDuplicates</code></td></tr>
<tr><td><b>Yêu cầu tài nguyên</b></td><td>Rất nhẹ, không tiêu tốn bộ nhớ lưu trữ trạng thái</td><td>Đòi hỏi State Store (HDFS / RocksDB) kết hợp Watermark để giải phóng bộ nhớ định kỳ</td></tr>
</tbody></table>

### 5.1. Các khái niệm cốt lõi: Event-Time, Windowing và Watermark

- **Thời gian sự kiện (Event-Time):** Thời điểm chính xác sự kiện phát sinh tại phía máy khách (thể hiện qua trường `timestamp` trong bản ghi), hoàn toàn khác biệt với *Processing-Time* (thời điểm hệ thống Spark nhận được dữ liệu). Một hệ thống Streaming ETL chuẩn mực bắt buộc phải căn cứ theo Event-Time để tránh sai lệch kết quả do độ trễ truyền thông.
- **Cửa sổ thời gian (Windowing):** Gom nhóm các sự kiện theo các khoảng thời gian xác định. Phổ biến gồm cửa sổ không chồng lấn (*Tumbling Window*), cửa sổ trượt (*Sliding Window*), và cửa sổ phiên (*Session Window*).
- **Cơ chế Watermark (`withWatermark`):** Là ngưỡng thời gian do kỹ sư thiết lập nhằm thông báo cho Spark Engine biết giới hạn tối đa mà hệ thống chấp nhận chờ đợi dữ liệu đến muộn (Late Data). Watermark cho phép engine **giải phóng an toàn trạng thái của các cửa sổ đã hoàn tất khỏi State Store**, hạn chế triệt để nguy cơ tràn bộ nhớ RAM (OOM) trong quá trình vận hành liên tục. Đề bài Part B đặt ra yêu cầu chính xác: *Cửa sổ trượt 10 phút, chu kỳ trượt 5 phút, ngưỡng Watermark 5 phút*.


<div style="overflow-x:auto; background:#ffffff; border:1px solid #e2e8f0; border-radius:12px; padding:18px 10px;">
<svg viewBox="0 0 860 300" width="100%" style="max-width:860px; font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;">
  <rect x="0" y="0" width="860" height="300" fill="#ffffff"/>
  <text x="430" y="26" text-anchor="middle" font-size="16" font-weight="700" fill="#0f172a">Sliding window 10&#8217; (trượt 5&#8217;) + Watermark 5&#8217;</text>

  <!-- time axis -->
  <line x1="40" y1="250" x2="820" y2="250" stroke="#334155" stroke-width="2" marker-end="url(#ah2)"/>
  <defs>
    <marker id="ah2" markerWidth="10" markerHeight="10" refX="7" refY="3" orient="auto" markerUnits="strokeWidth">
      <path d="M0,0 L7,3 L0,6 Z" fill="#334155"/>
    </marker>
  </defs>
  <text x="820" y="272" text-anchor="end" font-size="12" fill="#334155">event time</text>
  <!-- ticks -->
  <g font-size="11" fill="#64748b" text-anchor="middle">
    <line x1="120" y1="245" x2="120" y2="255" stroke="#64748b"/><text x="120" y="270">10:00</text>
    <line x1="280" y1="245" x2="280" y2="255" stroke="#64748b"/><text x="280" y="270">10:05</text>
    <line x1="440" y1="245" x2="440" y2="255" stroke="#64748b"/><text x="440" y="270">10:10</text>
    <line x1="600" y1="245" x2="600" y2="255" stroke="#64748b"/><text x="600" y="270">10:15</text>
    <line x1="760" y1="245" x2="760" y2="255" stroke="#64748b"/><text x="760" y="270">10:20</text>
  </g>

  <!-- windows -->
  <rect x="120" y="70" width="320" height="34" rx="6" fill="#dbeafe" stroke="#1e40af"/>
  <text x="280" y="92" text-anchor="middle" font-size="12" fill="#1e3a8a">W1 [10:00 &#8211; 10:10)</text>
  <rect x="280" y="112" width="320" height="34" rx="6" fill="#dcfce7" stroke="#166534"/>
  <text x="440" y="134" text-anchor="middle" font-size="12" fill="#166534">W2 [10:05 &#8211; 10:15)</text>
  <rect x="440" y="154" width="320" height="34" rx="6" fill="#fef3c7" stroke="#92400e"/>
  <text x="600" y="176" text-anchor="middle" font-size="12" fill="#92400e">W3 [10:10 &#8211; 10:20)</text>

  <!-- watermark line -->
  <line x1="600" y1="50" x2="600" y2="245" stroke="#dc2626" stroke-width="2" stroke-dasharray="6 4"/>
  <text x="608" y="60" font-size="12" font-weight="700" fill="#dc2626">Watermark = max(event_time) &#8722; 5&#8217;</text>

  <!-- late event -->
  <circle cx="330" cy="215" r="7" fill="#16a34a"/>
  <text x="330" y="235" text-anchor="middle" font-size="11" fill="#166534">event trễ &#10003; nhận</text>
  <circle cx="200" cy="215" r="7" fill="#dc2626"/>
  <text x="200" y="235" text-anchor="middle" font-size="11" fill="#b91c1c">quá trễ &#10007; bỏ</text>

  <text x="430" y="292" text-anchor="middle" font-size="11" fill="#94a3b8">Watermark cho phép state của window cũ được giải phóng, đồng thời vẫn chấp nhận dữ liệu đến muộn trong ngưỡng cho phép</text>
</svg>
</div>


### 5.2. Hiện thực logic nghiệp vụ của nhóm: Tính điểm ý định mua sắm (Intent Score)

Theo đặc tả nghiệp vụ của đề tài, nhóm tiến hành gán trọng số tương tác cho từng hành vi của người dùng: **view = 1, addtocart = 5, transaction = 10**, sau đó tính toán điểm ý định tích lũy (Intent Score) theo từng khách hàng (`visitorid`) trên cửa sổ trượt thời gian. Đây là bài toán **Stateful Transformation** điển hình trong thực tế thương mại điện tử.

In [5]:
# ===== TRANSFORM PHA 1 (STATELESS): Làm sạch dữ liệu và gán trọng số hành vi =====
# Thiết lập bảng tra trọng số tương tác theo từng loại sự kiện
weight_map = F.create_map(
    F.lit("view"),        F.lit(1),
    F.lit("addtocart"),   F.lit(5),
    F.lit("transaction"), F.lit(10),
)

clean_stream = (
    raw_stream
    # Chuyển đổi timestamp từ mili-giây sang TimestampType chuẩn của Spark để làm căn cứ Event-Time
    .withColumn("event_time", (F.col("timestamp") / 1000).cast(TimestampType()))
    .withColumn("weight", weight_map[F.col("event")])   # Gán trọng số điểm tương ứng
    .filter(F.col("event").isNotNull())                 # Lọc bỏ các bản ghi không hợp lệ
)

# ===== TRANSFORM PHA 2 (STATEFUL): Tính toán điểm tích lũy theo cửa sổ trượt kết hợp Watermark =====
intent_stream = (
    clean_stream
    .withWatermark("event_time", "5 minutes")   # Thiết lập ngưỡng Watermark 5 phút để xử lý dữ liệu đến muộn
    .groupBy(
        F.window("event_time", "10 minutes", "5 minutes").alias("w"),  # Cửa sổ trượt 10 phút, chu kỳ trượt 5 phút
        F.col("visitorid"),                                            # Gom nhóm theo từng mã khách truy cập
    )
    .agg(
        F.sum("weight").alias("intent_score"),      # Tổng điểm ý định mua sắm của khách trong cửa sổ
        F.count("*").alias("num_events"),           # Tổng số lượng sự kiện tương tác
        # Thống kê riêng tần suất thêm giỏ hàng và thanh toán phục vụ logic kích hoạt gợi ý
        F.sum(F.when(F.col("event") == "addtocart", 1).otherwise(0)).alias("addtocart_cnt"),
        F.sum(F.when(F.col("event") == "transaction", 1).otherwise(0)).alias("txn_cnt"),
    )
)
print("[Đồ thị tính toán] Trạng thái stream:", intent_stream.isStreaming, "| Kế hoạch biến đổi dữ liệu (Transform Plan) đã sẵn sàng.")

Streaming? -> True | đã dựng xong kế hoạch Transform (lazy).


Nhóm tiến hành thực thi truy vấn stateful trên tập dữ liệu hiện có bằng chế độ kích hoạt `trigger(availableNow=True)` (xử lý toàn bộ dữ liệu có sẵn rồi tạm dừng), nạp kết quả vào **Memory Sink** nhằm phục vụ quan sát và kiểm chứng kết quả tính toán:

In [6]:
# ===== THỰC THI TRUY VẤN STATEFUL ĐỐI SOÁT ĐIỂM Ý ĐỊNH MUA HÀNG =====
q_intent = (
    intent_stream.writeStream
    .format("memory").queryName("intent_tbl") # Ghi kết quả vào bảng tạm trong bộ nhớ mang tên intent_tbl
    .outputMode("complete")                   # Chế độ Complete xuất toàn bộ kết quả tổng hợp của bảng
    .trigger(availableNow=True)               # Xử lý toàn bộ micro-batch sẵn có và kết thúc truy vấn
    .start()
)
q_intent.awaitTermination()

# Truy vấn bảng trong bộ nhớ để trích xuất danh sách khách hàng có điểm ý định cao nhất
print(">>> [Kết quả thực nghiệm] Top khách hàng có điểm ý định (Intent Score) cao nhất:")
spark.sql("""
    SELECT visitorid,
           date_format(w.start, 'HH:mm') AS w_start,
           date_format(w.end,   'HH:mm') AS w_end,
           intent_score, num_events, addtocart_cnt, txn_cnt
    FROM intent_tbl
    ORDER BY intent_score DESC
    LIMIT 8
""").show(truncate=False)

>>> Điểm intent cao nhất theo (visitor, window):
+---------+-------+-----+------------+----------+-------------+-------+
|visitorid|w_start|w_end|intent_score|num_events|addtocart_cnt|txn_cnt|
+---------+-------+-----+------------+----------+-------------+-------+
|9        |12:15  |12:25|32          |10        |1            |2      |
|9        |12:10  |12:20|25          |7         |0            |2      |
|2        |12:20  |12:30|18          |6         |3            |0      |
|12       |12:30  |12:40|18          |5         |1            |1      |
|12       |12:25  |12:35|17          |4         |1            |1      |
|2        |12:35  |12:45|17          |5         |3            |0      |
|4        |12:15  |12:25|17          |8         |0            |1      |
|10       |12:05  |12:15|16          |4         |3            |0      |
+---------+-------+-----+------------+----------+-------------+-------+



### 5.3. Cơ chế kích hoạt gợi ý theo thời gian thực (Real-Time Recommendation Trigger)

**Quy tắc nghiệp vụ của nhóm:** Hệ thống thực hiện lọc các khách hàng thuộc nhóm **High-Intent** (điểm số Intent $\ge 5$), đặc biệt tập trung vào hành vi *đã đưa sản phẩm vào giỏ hàng (`addtocart_cnt > 0`) nhưng chưa hoàn tất thanh toán (`txn_cnt == 0`)* để kích hoạt hành động gợi ý tức thì. 

Đây chính là **điểm kết nối kiến trúc then chốt giữa tầng Streaming ETL và Tầng Gợi ý (Recommender System ở Part A.2)**: Tầng ETL đóng vai trò tiền xử lý và cung cấp chính xác các "tín hiệu tương tác có trọng số" trong phiên để hệ thống gợi ý đưa ra danh mục sản phẩm cá nhân hóa phù hợp nhất cho người dùng.

In [7]:
# ===== PHÂN LỚP Ý ĐỊNH: KÍCH HOẠT HÀNH ĐỘNG GỢI Ý THỜI GIAN THỰC =====
reco_df = spark.sql("""
    SELECT visitorid,
           date_format(w.start, 'HH:mm') AS w_start,
           intent_score, addtocart_cnt, txn_cnt,
           CASE WHEN addtocart_cnt > 0 AND txn_cnt = 0
                THEN 'GỢI Ý NGAY (đã add-to-cart, chưa mua)'
                ELSE 'tiếp tục theo dõi' END AS action
    FROM intent_tbl
    WHERE intent_score >= 5        -- Lọc khách hàng có mức độ tương tác cao
    ORDER BY intent_score DESC
    LIMIT 10
""")
reco_df.show(truncate=False)

+---------+-------+------------+-------------+-------+-------------------------------------+
|visitorid|w_start|intent_score|addtocart_cnt|txn_cnt|action                               |
+---------+-------+------------+-------------+-------+-------------------------------------+
|9        |12:15  |32          |1            |2      |theo dõi tiếp                        |
|9        |12:10  |25          |0            |2      |theo dõi tiếp                        |
|2        |12:20  |18          |3            |0      |GỢI Ý NGAY (đã add-to-cart, chưa mua)|
|12       |12:30  |18          |1            |1      |theo dõi tiếp                        |
|12       |12:25  |17          |1            |1      |theo dõi tiếp                        |
|2        |12:35  |17          |3            |0      |GỢI Ý NGAY (đã add-to-cart, chưa mua)|
|4        |12:15  |17          |0            |1      |theo dõi tiếp                        |
|3        |12:05  |16          |0            |1      |theo dõi tiếp   

## 6. Khả năng chịu lỗi (Fault Tolerance) — Write-Ahead Log (WAL) và Checkpointing

Trong môi trường phân tán quy mô lớn, một vấn đề kiến trúc trọng tâm là: *Hệ thống phản ứng ra sao khi một Worker Node hoặc Driver gặp sự cố đột ngột?* Spark Structured Streaming giải quyết triệt để bài toán này thông qua cơ chế **Checkpointing** kết hợp **Write-Ahead Log (WAL)** — việc cấu hình thuộc tính `checkpointLocation` là bắt buộc đối với mọi truy vấn streaming trong môi trường production. 

Cấu trúc thư mục Checkpoint được chuẩn hóa bao gồm các thành phần:

| Thành phần thư mục | Chức năng kỹ thuật trong kiến trúc chịu lỗi |
|---|---|
| **offsets/** | **Write-Ahead Log (WAL):** Ghi nhận *trước* phạm vi offset dự kiến xử lý của mỗi micro-batch (ghi nhật ký bền vững trước khi phân phối công việc cho worker) |
| **commits/** | Đánh dấu micro-batch đã hoàn tất xử lý thành công và sink đã ghi nhận dữ liệu |
| **state/** | Lưu trữ ảnh chụp nhanh (State Store Snapshot) có đánh số phiên bản theo từng batch cho các phép toán stateful |
| **sources/** | Quản lý siêu dữ liệu nguồn (ví dụ: danh sách định danh tệp đã được đọc đối với File Stream) |
| **metadata** | Lưu trữ mã định danh duy nhất của truy vấn streaming |

**Cơ chế tự phục hồi sau sự cố (Recovery Workflow):** Khi tiến trình được khởi động lại, Spark Engine sẽ đọc nhật ký `offsets/` để xác định micro-batch đang xử lý dở dang, đối chiếu với nhật ký `commits/` để xác thực trạng thái ghi nhận, khôi phục trạng thái từ `state/` đúng phiên bản tương ứng, và tiến hành **thực thi lại chính xác batch bị gián đoạn** mà không gây thất thoát dữ liệu.

Dưới đây là mã nguồn nạp dữ liệu vào định dạng lưu trữ cột Parquet có kích hoạt checkpoint, kết hợp trích xuất trực quan cấu trúc của Write-Ahead Log:

In [8]:
# ===== LOAD: Nạp dữ liệu vào định dạng Parquet kèm ghi nhật ký có cấu trúc và Checkpointing =====
CKPT    = f"{BASE}/_checkpoint_curated"    # Thư mục lưu trữ Checkpoint (WAL + Commits + State)
CURATED = f"{BASE}/curated_parquet"        # Thư mục lưu trữ dữ liệu đã làm sạch (Tầng Curated Parquet)
shutil.rmtree(CKPT, ignore_errors=True)
shutil.rmtree(CURATED, ignore_errors=True)

# Triển khai hàm foreachBatch để xử lý và ghi log có cấu trúc cho từng micro-batch độc lập
def process_batch(batch_df, batch_id):
    t0 = time.time()
    n = batch_df.count()                                 # Thống kê tổng số bản ghi trong micro-batch
    (batch_df.write.mode("append")                       # Ghi dữ liệu theo cơ chế nối tiếp (Append)
             .partitionBy("event")                       # Phân vùng vật lý trên đĩa theo loại sự kiện để tối ưu truy vấn
             .parquet(CURATED))
    # Ghi nhận log có cấu trúc: ID batch, số lượng bản ghi, thời gian thực thi
    print(f"[BATCH {batch_id}] records={n:<4d} elapsed={time.time()-t0:0.2f}s "
          f"-> Đã nạp vào Parquet (phân vùng theo trường event)")

q_curated = (
    clean_stream.select("event_time", "visitorid", "event", "itemid", "weight")
    .writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", CKPT)          # BẬT cơ chế Checkpointing để kích hoạt WAL
    .trigger(availableNow=True)
    .start()
)
q_curated.awaitTermination()
print("\n[Hoàn thành] Đã hoàn tất chu trình nạp dữ liệu vào tầng Parquet Curated.")

[BATCH 0] records=60   elapsed=0.80s -> ghi Parquet (partitioned by event)


[BATCH 1] records=60   elapsed=0.15s -> ghi Parquet (partitioned by event)


[BATCH 2] records=60   elapsed=0.14s -> ghi Parquet (partitioned by event)

Đã nạp xong vào Parquet curated.


In [9]:
# ===== KIỂM TRA CẤU TRÚC CHECKPOINT VÀ NHẬT KÝ WRITE-AHEAD LOG (WAL) =====
print("[Hệ thống] Cấu trúc các thư mục thành phần bên trong Checkpoint:")
for name in sorted(os.listdir(CKPT)):
    print("  ├─", name)

def first_file(sub):
    p = os.path.join(CKPT, sub)
    fs = [f for f in sorted(os.listdir(p)) if not f.startswith(".")] if os.path.isdir(p) else []
    return os.path.join(p, fs[0]) if fs else None

off = first_file("offsets")                  # Tệp nhật ký đầu tiên trong offsets/ chính là Write-Ahead Log (WAL)
if off:
    print(f"\n--- [Minh chứng WAL] Nội dung offset log của batch đầu tiên ({os.path.basename(off)}) ---")
    print(open(off).read().strip())
print("\n[Nhật ký commits/]:", os.listdir(os.path.join(CKPT, "commits")))

Nội dung thư mục checkpoint:
  ├─ .metadata.crc
  ├─ commits
  ├─ metadata
  ├─ offsets
  ├─ sources

--- Nội dung offset log batch đầu tiên (0) = WAL ---
v1
{"batchWatermarkMs":0,"batchTimestampMs":1793030944483,"conf":{"spark.sql.streaming.stateStore.providerClass":"org.apache.spark.sql.execution.streaming.state.HDFSBackedStateStoreProvider","spark.sql.streaming.join.stateFormatVersion":"2","spark.sql.streaming.stateStore.compression.codec":"lz4","spark.sql.optimizer.pruneFiltersCanPruneStreamingSubplan":"false","spark.sql.streaming.stateStore.rocksdb.formatVersion":"5","spark.sql.streaming.statefulOperator.useStrictDistribution":"true","spark.sql.streaming.flatMapGroupsWithState.stateFormatVersion":"2","spark.sql.streaming.multipleWatermarkPolicy":"min","spark.sql.streaming.aggregation.stateFormatVersion":"2","spark.sql.shuffle.partitions":"4"}}
{"logOffset":0}

commits/: ['.2.crc', '.0.crc', '.1.crc', '0', '1', '2']


In [10]:
# ===== ĐỐI SOÁT VÀ KIỂM CHỨNG DỮ LIỆU ĐÃ NẠP (LOAD VERIFICATION) =====
curated = spark.read.parquet(CURATED)        # Đọc dữ liệu đã nạp ở chế độ Batch thông thường
print("[Thống kê] Tổng số lượng bản ghi tại tầng Curated:", curated.count())
curated.groupBy("event").count().orderBy(F.desc("count")).show()
print("[Cấu trúc phân vùng vật lý trên đĩa (partitionBy event)]:")
for d in sorted(glob.glob(f"{CURATED}/event=*")):
    print("  ", os.path.basename(d))

Tổng bản ghi curated: 180
+-----------+-----+
|      event|count|
+-----------+-----+
|       view|  151|
|  addtocart|   22|
|transaction|    7|
+-----------+-----+

Phân vùng trên đĩa (partitionBy event):
   event=addtocart
   event=transaction
   event=view


## 7. Cơ chế đảm bảo ngữ nghĩa End-to-End Exactly-Once

Trong xử lý dữ liệu dòng phân tán, câu hỏi trọng tâm về độ tin cậy là: *Khi sự cố xảy ra và hệ thống tái khởi động, liệu dữ liệu có bị thất thoát hoặc bị tính toán trùng lặp hay không?* Ngữ nghĩa **Exactly-once** bảo đảm rằng mỗi sự kiện chỉ tác động đến kết quả đầu ra cuối cùng **đúng một lần duy nhất**, ngay cả khi xảy ra lỗi phần cứng hoặc gián đoạn mạng.

**So sánh các cấp độ đảm bảo ngữ nghĩa xử lý:**

<table>
<thead><tr><th>Cấp độ ngữ nghĩa</th><th>Đặc tả kỹ thuật</th><th>Hạn chế / Rủi ro</th></tr></thead>
<tbody>
<tr><td><b>At-most-once</b> (Tối đa một lần)</td><td>Mỗi bản ghi được xử lý tối đa 1 lần, không phát lại khi gặp lỗi</td><td>Nguy cơ <b>thất thoát dữ liệu</b> cao</td></tr>
<tr><td><b>At-least-once</b> (Ít nhất một lần)</td><td>Bản ghi được phát lại và xử lý lại khi có lỗi</td><td>Nguy cơ <b>trùng lặp bản ghi</b> tại hệ thống đích</td></tr>
<tr><td><b>Exactly-once</b> (Đúng một lần)</td><td>Mọi bản ghi đều được xử lý với kết quả tương đương như chỉ chạy một lần duy nhất</td><td>Đòi hỏi sự phối hợp chặt chẽ giữa Source, Engine và Sink</td></tr>
</tbody></table>

**Ba điều kiện tiên quyết để Spark Structured Streaming đạt được ngữ nghĩa End-to-End Exactly-Once:**

1. **Nguồn dữ liệu có khả năng phát lại (Replayable Source):** Hệ thống nguồn (như Apache Kafka hoặc File Stream) phải có cơ chế ghi nhớ vị trí con trỏ (Offset) chính xác, cho phép phát lại dữ liệu từ một mốc thời gian/offset bất kỳ khi có sự cố.
2. **Cơ chế quản lý tiến độ xác định (Deterministic Checkpointing & WAL):** Spark engine bắt buộc phải ghi lại phạm vi offset chuẩn bị xử lý vào Write-Ahead Log trước khi phân phối công việc cho các worker, đồng thời lưu trữ snapshot trạng thái (State Store) theo phiên bản tương ứng của từng batch.
3. **Đích nạp có tính chất Idempotent (Idempotent / Transactional Sink):** Hệ thống lưu trữ đích (như Parquet kết hợp Commit Log, Delta Lake với ACID transactions, hoặc các hệ thống hỗ trợ upsert theo khóa định danh) phải đảm bảo việc ghi lại một batch đã commit sẽ không tạo ra các bản ghi trùng lặp.

Sự kết hợp đồng bộ giữa **Replayable Source + WAL & State Checkpointing + Idempotent Sink** tạo nên kiến trúc chịu lỗi toàn diện, bảo đảm tính toàn vẹn dữ liệu cho toàn bộ pipeline.


<div style="overflow-x:auto; background:#ffffff; border:1px solid #e2e8f0; border-radius:12px; padding:18px 10px;">
<svg viewBox="0 0 860 280" width="100%" style="max-width:860px; font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;">
  <rect x="0" y="0" width="860" height="280" fill="#ffffff"/>
  <text x="430" y="26" text-anchor="middle" font-size="16" font-weight="700" fill="#0f172a">Cơ chế end-to-end exactly-once</text>
  <defs>
    <marker id="ah3" markerWidth="10" markerHeight="10" refX="7" refY="3" orient="auto" markerUnits="strokeWidth">
      <path d="M0,0 L7,3 L0,6 Z" fill="#475569"/>
    </marker>
  </defs>

  <!-- source -->
  <rect x="30" y="80" width="170" height="90" rx="10" fill="#dbeafe" stroke="#1e40af"/>
  <text x="115" y="108" text-anchor="middle" font-size="13" font-weight="700" fill="#1e3a8a">1. Replayable source</text>
  <text x="115" y="130" text-anchor="middle" font-size="11" fill="#1e3a8a">Kafka offsets / file log</text>
  <text x="115" y="148" text-anchor="middle" font-size="11" fill="#1e3a8a">đọc lại được khi fail</text>

  <!-- engine + checkpoint -->
  <rect x="250" y="70" width="200" height="110" rx="10" fill="#dcfce7" stroke="#166534"/>
  <text x="350" y="96" text-anchor="middle" font-size="13" font-weight="700" fill="#166534">2. Engine + Checkpoint</text>
  <text x="350" y="118" text-anchor="middle" font-size="11" fill="#166534">offset log (WAL) ghi trước</text>
  <text x="350" y="136" text-anchor="middle" font-size="11" fill="#166534">state store có phiên bản</text>
  <text x="350" y="154" text-anchor="middle" font-size="11" fill="#166534">commit log sau khi xong</text>

  <!-- sink -->
  <rect x="500" y="80" width="170" height="90" rx="10" fill="#fef3c7" stroke="#92400e"/>
  <text x="585" y="108" text-anchor="middle" font-size="13" font-weight="700" fill="#92400e">3. Idempotent sink</text>
  <text x="585" y="130" text-anchor="middle" font-size="11" fill="#92400e">manifest / transaction</text>
  <text x="585" y="148" text-anchor="middle" font-size="11" fill="#92400e">bỏ batch đã commit</text>

  <!-- result -->
  <rect x="712" y="80" width="118" height="90" rx="10" fill="#ede9fe" stroke="#6d28d9"/>
  <text x="771" y="120" text-anchor="middle" font-size="13" font-weight="700" fill="#5b21b6">Kết quả</text>
  <text x="771" y="142" text-anchor="middle" font-size="12" fill="#5b21b6">đúng 1 lần</text>

  <line x1="202" y1="125" x2="246" y2="125" stroke="#475569" stroke-width="2.5" marker-end="url(#ah3)"/>
  <line x1="452" y1="125" x2="496" y2="125" stroke="#475569" stroke-width="2.5" marker-end="url(#ah3)"/>
  <line x1="672" y1="125" x2="708" y2="125" stroke="#475569" stroke-width="2.5" marker-end="url(#ah3)"/>

  <text x="430" y="212" text-anchor="middle" font-size="12" fill="#334155">Khi job crash giữa chừng: Spark đọc lại <tspan font-weight="700">offset log</tspan> để biết batch dở dang, phục hồi <tspan font-weight="700">state</tspan> từ checkpoint,</text>
  <text x="430" y="232" text-anchor="middle" font-size="12" fill="#334155">xử lý lại đúng batch đó; sink dựa vào <tspan font-weight="700">commit log</tspan> để không ghi trùng → hiệu lực <tspan font-weight="700">exactly-once</tspan>.</text>
</svg>
</div>


> **Định hướng kiến trúc cho Part B:** Việc kết hợp **Kafka Source + Parquet Sink + `checkpointLocation`** cung cấp cho hệ thống của nhóm cơ chế đảm bảo ngữ nghĩa End-to-End Exactly-Once hoàn chỉnh. Đây là nguyên tắc cốt lõi giải thích vì sao mọi tác vụ `writeStream` trong môi trường triển khai thực tế đều phải có định danh checkpoint độc lập. (Lưu ý: `ConsoleSink` không hỗ trợ phục hồi và không có tính idempotent, do đó chỉ phù hợp trong giai đoạn gỡ lỗi sơ bộ).

**Kiểm chứng thực nghiệm tính Idempotent:**  
Nhóm tiến hành tái thực thi truy vấn trên cùng một thư mục checkpoint. Kết quả kiểm chứng xác nhận số lượng bản ghi không hề gia tăng, do Spark nhận biết các batch này đã hoàn tất trong commit log:

In [11]:
# ===== KIỂM CHỨNG TÍNH CHẤT IDEMPOTENT: Thực thi lại để chứng minh không trùng lặp dữ liệu =====
count_before = spark.read.parquet(CURATED).count()   # Số lượng bản ghi trước khi tái thực thi

def process_batch_again(batch_df, batch_id):
    print(f"[RERUN BATCH {batch_id}] records={batch_df.count()}")
    batch_df.write.mode("append").partitionBy("event").parquet(CURATED)

q_rerun = (
    clean_stream.select("event_time", "visitorid", "event", "itemid", "weight")
    .writeStream.foreachBatch(process_batch_again)
    .option("checkpointLocation", CKPT)          # Tái sử dụng Checkpoint cũ -> Spark nhận biết dữ liệu đã được xử lý
    .trigger(availableNow=True).start()
)
q_rerun.awaitTermination()

count_after = spark.read.parquet(CURATED).count()    # Số lượng bản ghi sau khi tái thực thi
print(f"\n[Kiểm chứng] Trước khi chạy lại: {count_before} bản ghi | Sau khi chạy lại: {count_after} bản ghi")
print("=> Kết luận: Tổng số bản ghi giữ nguyên tuyệt đối, hệ thống không bị ghi trùng dữ liệu (đạt tính chất Idempotent).")


Trước rerun: 180 | Sau rerun: 180
=> Bằng nhau: offset log cho biết mọi file đã xử lý, KHÔNG ghi trùng (idempotent).


## 8. Tổng hợp: Mô hình kiến trúc dữ liệu theo phân tầng (Medallion Architecture)

Trong các hệ thống Big Data hiện đại, mô hình **Kiến trúc Medallion (Multi-Hop Architecture)** được áp dụng rộng rãi để chuẩn hóa dữ liệu dòng qua 3 tầng chất lượng tăng dần:

<table>
<thead><tr><th>Tầng dữ liệu</th><th>Ý nghĩa và vai trò kiến trúc</th><th>Ánh xạ vào bài toán của nhóm</th></tr></thead>
<tbody>
<tr><td><b>🥉 Bronze (Raw Layer)</b></td><td>Lưu trữ dữ liệu thô nguyên bản từ các hệ thống thu thập, bảo đảm tính bất biến</td><td>Nhận luồng JSON telemetry từ Kafka topic và lưu trữ trực tiếp</td></tr>
<tr><td><b>🥈 Silver (Curated Layer)</b></td><td>Dữ liệu đã qua lọc nhiễu, chuẩn hóa kiểu dữ liệu, làm sạch và gắn nhãn thuộc tính</td><td>Phân giải timestamp, gán trọng số hành vi (weight map), lọc bỏ bản ghi bất hợp lệ</td></tr>
<tr><td><b>🥇 Gold (Aggregated Layer)</b></td><td>Dữ liệu tổng hợp mức cao, phục vụ trực tiếp cho báo cáo phân tích và các mô hình học máy</td><td>Tính toán điểm Intent Score theo cửa sổ trượt, cung cấp đầu vào cho hệ thống gợi ý</td></tr>
</tbody></table>

Mô hình phân tầng trên phản ánh trọn vẹn chu trình E → T → L mà nhóm đã thiết kế. Đoạn mã dưới đây minh chứng quy trình phân loại và ghi nhận trạng thái đồng thời của 3 tầng dữ liệu qua mỗi micro-batch:

In [12]:
# ===== TỔNG KẾT: Phân tách và ghi nhận trạng thái 3 tầng Bronze/Silver/Gold qua từng micro-batch =====
SUMMARY_CKPT = f"{BASE}/_checkpoint_summary"
shutil.rmtree(SUMMARY_CKPT, ignore_errors=True)

def medallion_log(batch_df, batch_id):
    bronze = batch_df.count()                                       # Bronze: Dữ liệu thô nguyên bản
    silver = batch_df.filter(F.col("event").isNotNull()).count()    # Silver: Dữ liệu đã làm sạch
    # Gold: Số lượng khách hàng có điểm ý định tích lũy >= 5
    gold   = (batch_df.groupBy("visitorid").agg(F.sum("weight").alias("s"))
                      .filter(F.col("s") >= 5).count())
    print(f"[Micro-batch {batch_id}] Bronze(Raw)={bronze:<4d} "
          f"Silver(Clean)={silver:<4d} Gold(High-Intent Visitors)={gold}")

q_sum = (
    clean_stream
    .writeStream.foreachBatch(medallion_log)
    .option("checkpointLocation", SUMMARY_CKPT)
    .trigger(availableNow=True).start()
)
q_sum.awaitTermination()
print("\n[Kết luận] Mỗi micro-batch đều được xử lý trọn vẹn qua các pha Extract -> Transform -> Load với đầy đủ cơ chế Logging và Checkpointing.")

[micro-batch 0] Bronze(raw)=60   Silver(clean)=60   Gold(high-intent visitors)=8


[micro-batch 1] Bronze(raw)=60   Silver(clean)=60   Gold(high-intent visitors)=8


[micro-batch 2] Bronze(raw)=60   Silver(clean)=60   Gold(high-intent visitors)=9

Mỗi micro-batch đều đi qua đủ Extract -> Transform -> Load, có logging + checkpoint.


## 9. Tổng quan tài liệu nghiên cứu (Literature Review) về tối ưu hóa Streaming ETL

> **Trích dẫn chuẩn mực (APA):**  
> Armbrust, M., Das, T., Sun, S., Yavuz, B., Zhu, S., Murthy, M., Torres, J., van Hovell, H., Ionescu, R., Luo, X., Shen, X., Xiao, L., Lian, M., Gu, L., Dilipkumar, B., & Zaharia, M. (2018). *Structured Streaming: A Declarative API for Real-Time Applications in Apache Spark.* Proceedings of the 2018 International Conference on Management of Data (SIGMOD '18), 601–613. ACM. https://doi.org/10.1145/3183713.3190664

### 9.1. Bài toán và bối cảnh nghiên cứu
Trước khi Structured Streaming ra đời, các mô hình xử lý dòng thế hệ trước (điển hình như Spark DStream API dựa trên kiến trúc RDD cấp thấp) đặt ra những thách thức lớn đối với kỹ sư phát triển:
- Lập trình viên phải tự quản lý việc phân chia vi lô thủ công và tự xây dựng logic phức tạp nhằm xử lý dữ liệu đến muộn (Late / Out-of-Order Data).
- Sự phân mảnh giữa hai mô hình lập trình riêng biệt: Xử lý theo lô (Batch Processing) sử dụng DataFrame/SQL trong khi xử lý dòng (Stream Processing) phải chuyển đổi sang DStream API, làm gia tăng chi phí bảo trì và kiểm thử.
- Khó khăn trong việc phối hợp trạng thái để bảo đảm tính nhất quán dữ liệu và đạt được ngữ nghĩa *Exactly-Once* khi có sự cố hệ thống xảy ra.

### 9.2. Các đóng góp khoa học và kỹ thuật cốt lõi của công trình
1. **Mô hình lập trình khai báo trừu tượng (Declarative API & Unbounded Table Model):**  
   Dòng dữ liệu được trừu tượng hóa dưới dạng một "Bảng vô hạn" (Unbounded Table) liên tục nhận dữ liệu mới. Nhờ đó, cùng một tập lệnh logic truy vấn DataFrame/SQL duy nhất có thể được áp dụng một cách đồng nhất cho cả dữ liệu tĩnh và dữ liệu dòng, loại bỏ sự phân mảnh kiến trúc.
2. **Tự động tối ưu hóa qua Catalyst Optimizer và Tungsten Engine:**  
   Thay vì lập trình viên phải tự tối ưu các phép toán RDD cấp thấp, Spark Engine tự động sinh kế hoạch thực thi gia tăng (Incremental Execution Plan), tối ưu hóa việc quản lý bộ nhớ và biên dịch mã thực thi (Code Generation) tốc độ cao.
3. **Cơ chế quản lý trạng thái tự động và chịu lỗi nghiêm ngặt:**  
   Hệ thống tích hợp sẵn cơ chế thiết lập ngưỡng Watermark theo thời gian thực tế của sự kiện (Event-Time), kết hợp với State Store có phiên bản và Write-Ahead Log để bảo đảm ngữ nghĩa *End-to-End Exactly-Once* hoàn chỉnh mà không đòi hỏi can thiệp thủ công vào tầng lưu trữ.
4. **Hiệu năng thực nghiệm vượt trội:**  
   Công bố thực nghiệm trong bài báo chứng minh Structured Streaming đạt thông lượng xử lý cao hơn và độ trễ thấp hơn (trong nhiều kịch bản thử nghiệm đạt hiệu năng xấp xỉ gấp đôi so với các nền tảng cùng thời như Apache Flink đối với các tác vụ tổng hợp và biến đổi quy mô lớn).

### 9.3. Ý nghĩa đối với đề tài nghiên cứu của nhóm
Công trình nghiên cứu của Armbrust et al. (2018) cung cấp nền tảng cơ sở lý thuyết vững chắc cho kiến trúc mà nhóm triển khai trong đề tài:
- Khẳng định tính đúng đắn và tính khoa học của việc áp dụng mô hình Declarative Streaming DataFrame cho bài toán phân tích hành vi phiên truy cập (Session Tracker) của Retail Rocket.
- Đảm bảo rằng pipeline ETL do nhóm thiết kế tại Part B không chỉ đạt thông lượng cao và độ trễ thấp, mà còn duy trì tính toán chính xác tuyệt đối nhờ cơ chế Checkpointing và Idempotent Sink được chuẩn hóa theo đúng khuyến nghị của các nhà nghiên cứu Apache Spark.

## 10. Tổng kết & Đối chiếu yêu cầu đề bài với Part B

Bảng đối chiếu việc hoàn thành các mục tiêu nghiên cứu của nhóm theo tiêu chí đề bài:

<table>
<thead><tr><th>Yêu cầu trọng tâm của đề bài (Part A — ETL)</th><th>Nội dung giải quyết và chứng minh trong Notebook</th></tr></thead>
<tbody>
<tr><td><b>1. Tầm quan trọng của ETL trong xử lý liên tục</b></td><td>Phân tích tại Mục 1 (So sánh chi tiết Batch vs Streaming ETL, ETL vs ELT)</td></tr>
<tr><td><b>2. Schema Enforcement</b></td><td>Hiện thực tại Mục 4 (Khai báo lược đồ tường minh, áp dụng <code>from_json</code>, kiểm chứng cô lập bản ghi lỗi)</td></tr>
<tr><td><b>3. Stateful Transformations</b></td><td>Hiện thực tại Mục 5 (Cấu hình Sliding Window 10 phút, Watermark 5 phút, thuật toán tính điểm Intent Score)</td></tr>
<tr><td><b>4. Fault Tolerance (WAL + Checkpointing)</b></td><td>Hiện thực tại Mục 6 (Trích xuất và kiểm tra thực tế cấu trúc Write-Ahead Log và Commit Log)</td></tr>
<tr><td><b>5. End-to-End Exactly-Once Semantics</b></td><td>Phân tích và chứng minh tại Mục 7 (Thực nghiệm tái thực thi khẳng định tính chất Idempotent)</td></tr>
<tr><td><b>6. Tổng quan tài liệu nghiên cứu (Literature)</b></td><td>Trình bày tại Mục 9 (Tóm tắt và liên hệ công trình SIGMOD 2018 của Armbrust et al.)</td></tr>
</tbody></table>

**Cầu nối triển khai thực tế sang Part B:**  
Toàn bộ logic xử lý ở tầng Transform mà nhóm đã kiểm chứng trong notebook này (bao gồm giải mã `from_json`, chuẩn hóa `event_time`, bảng tra trọng số `weight_map`, cửa sổ `window` kết hợp `withWatermark`, tính điểm Intent Score, hàm ghi log `foreachBatch`, nạp dữ liệu vào Parquet kèm `checkpointLocation`) được tích hợp nguyên vẹn vào module chính `Part_B/streaming_etl.py`. Điểm khác biệt duy nhất khi chuyển sang Part B là **thay đổi tầng trích xuất (Extract): chuyển từ File Streaming Source sang Kafka Source phân tán** (`spark.readStream.format("kafka")`), trong khi toàn bộ tầng Transform và Load được kế thừa hoàn toàn.

### Tài liệu tham khảo chính
1. Armbrust, M., et al. (2018). *Structured Streaming: A Declarative API for Real-Time Applications in Apache Spark.* Proceedings of the 2018 International Conference on Management of Data (SIGMOD '18), 601–613. https://doi.org/10.1145/3183713.3190664
2. Apache Spark Organization. (2024). *Structured Streaming Programming Guide.* Apache Spark Official Documentation. https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html
3. Zaharia, M., et al. (2013). *Discretized Streams: Fault-Tolerant Streaming Computation at Scale.* Proceedings of the 24th ACM Symposium on Operating Systems Principles (SOSP '13).

In [13]:
# ===== GIẢI PHÓNG TÀI NGUYÊN HỆ THỐNG =====
# Đóng phiên làm việc SparkSession sau khi hoàn thành toàn bộ nội dung thực nghiệm
spark.stop()
print("[Hoàn tất] Đã dừng SparkSession. Hoàn thành toàn bộ báo cáo chuyên đề Part A.1.2: ETL Data Architecture in Spark Streaming.")

Đã dừng SparkSession. Xong phần ETL Data Architecture in Spark Streaming.
